# Preparació dels jobs de Rosetta Relax — IL-7Rα

Aquest notebook prepara i envia al clúster els càlculs d'optimització energètica amb Rosetta Relax per als complexos target–binder del dataset IL-7Rα. Com a inputs s'utilitzen els fitxers .pdb dels millors models d'AlphaFold3 (seleccionats per ranking_score), que s'han traslladat prèviament a la carpeta af3_pdbs_il7ra.

La classe proteinModels de la llibreria prepare_proteins carrega els models des de la carpeta de PDBs. La funció setUpRosettaOptimization crea l'estructura de carpetes necessària i genera les comandes per executar Rosetta Relax amb els paràmetres nstruct=111 (nombre d'estructures optimitzades per complex) i cst_optimization=False. Les cadenes es tracten sense retallar els extrems terminals per tal de preservar la integritat de la interfície d'interacció.

Els requeriments de submissió al SLURM del MN5 s'especifiquen amb ntasks=112 (un per CPU del node) i cpus_per_task=1, que permeten ocupar un node sencer i aprofitar la velocitat de comunicació intra-node. Els càlculs s'envien a la partició gp_bscls.

In [ ]:
import sys
from pathlib import Path
from prepare_proteins import proteinModels
from bsc_calculations import mn5

base = Path.cwd()  # Carpeta que correspon a Scripts (path fins on tinc aquest codi)
pdb_folder = base / "af3_pdbs_il7ra"   # Carpeta on es troben els .pdb
relax_folder = base / "rosetta_relax_il7ra"

models_obj = proteinModels(str(pdb_folder))

print("Models carregats:")
print(models_obj.models_names)

rosetta_jobs = models_obj.setUpRosettaOptimization(
    relax_folder=str(relax_folder),
    nstruct=111,
    cst_optimization=False,
)

print(f"\nS'han preparat {len(rosetta_jobs)} jobs de Rosetta.")

if rosetta_jobs:
    mn5.jobArrays(
        rosetta_jobs,
        ntasks=112,
        cpus_per_task=1,
        program="rosetta",
        job_name="rosetta_il7ra",
        partition="gp_bscls",
    )